<a href="https://colab.research.google.com/github/arjunanlokeshwaran7-droid/Agentic-AI/blob/main/23it35.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-community langchain-text-splitters
!pip install sentence-transformers faiss-cpu chromadb
!pip install groq pypdf pymupdf

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Resume 35.pdf to Resume 35.pdf


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("Resume 35.pdf")
documents = loader.load()

The previous error indicated that no text could be extracted from the PDF. This often happens if the PDF is image-based (e.g., a scan) and doesn't contain selectable text. To address this, we will use `PyMuPDF` (imported as `fitz`), a more robust library for PDF processing that can extract text even from image-based PDFs, potentially utilizing OCR internally.

In [ ]:
import fitz
from langchain_core.documents import Document

def extract_text_from_pdf_with_pymupdf(pdf_path):
    doc = fitz.open(pdf_path)
    extracted_documents = []
    for page_num in range(doc.page_count):
        page = doc.load_page(page_num)
        text = page.get_text("text")
        if text.strip():
            extracted_documents.append(Document(page_content=text, metadata={"page": page_num + 1, "source": pdf_path}))
    return extracted_documents


pdf_file_path = "Resume 35.pdf"
documents = extract_text_from_pdf_with_pymupdf(pdf_file_path)

print(f"Extracted {len(documents)} pages from {pdf_file_path}")
if documents:
    print("First 200 characters of the first page:")
    print(documents[0].page_content[:200])
else:
    print("No text extracted from the PDF.")

Extracted 1 pages from Resume 35.pdf
First 200 characters of the first page:
Is an AI-powered web app that assesses
mental health and provides personalized
therapy suggestions.
AI-based face recognition leverages deep
learning and computer vision to detect and
authenticate fac


In [ ]:
from langchain_text_splitters import CharacterTextSplitter

char_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=200,
    chunk_overlap=20
)

char_docs = char_splitter.split_documents(documents)


for i, chunk in enumerate(char_docs[:10]):
    print(f"\n🔹 Chunk {i+1}:")
    print(chunk.page_content)
    print("-" * 50)

print("Total chunks:", len(char_docs))


🔹 Chunk 1:
Is an AI-powered web app that assesses
mental health and provides personalized
therapy suggestions.
AI-based face recognition leverages deep
learning and computer vision to detect and
--------------------------------------------------

🔹 Chunk 2:
authenticate faces in images or videos.
This AI Cost Calculator estimates AI usage
costs based on inputs like API calls and
tokens.
It helps teams plan budgets and manage AI
expenses effectively.
--------------------------------------------------

🔹 Chunk 3:
I am a responsible and adaptable person with
good communication skills and a positive attitude.
I am quick to learn, hardworking, and capable of
working both independently and as part of a
--------------------------------------------------

🔹 Chunk 4:
team. My goal is to gain practical experience and
grow along with the organization.
2023 TO 2025 
SSLC / HSC
Vidhya vikas martic hr.sec
schol 
with 75.86 
Programming Languages and Algorithms
--------------------------------------

Since even PyMuPDF couldn't extract text, the PDF is likely entirely image-based (e.g., a scanned document). We will now use Optical Character Recognition (OCR) to extract text from the images within the PDF. We'll use `Tesseract` OCR, along with its Python wrapper `pytesseract` and `Pillow` for image processing.

In [ ]:
!apt-get update
!apt-get install -y tesseract-ocr
!pip install pytesseract pillow

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 

In [ ]:
import fitz  # PyMuPDF
import pytesseract
from PIL import Image
from langchain_core.documents import Document
import io

def extract_text_with_ocr_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    extracted_documents = []
    for page_num in range(doc.page_count):
        page = doc.load_page(page_num)

        text = page.get_text("text")

        if not text.strip():
            print(f"Applying OCR to page {page_num + 1}...")

            pix = page.get_pixmap(dpi=300)
            img_byte_arr = pix.tobytes("png")
            img = Image.open(io.BytesIO(img_byte_arr))

            try:
                ocr_text = pytesseract.image_to_string(img)
                text = ocr_text
            except pytesseract.TesseractNotFoundError:
                print("Tesseract is not installed or not in PATH. Please install it.")
                text = ""
            except Exception as e:
                print(f"OCR failed for page {page_num + 1}: {e}")
                text = ""

        if text.strip():
            extracted_documents.append(Document(page_content=text, metadata={"page": page_num + 1, "source": pdf_path}))
    return extracted_documents

pdf_file_path = "Resume 35.pdf"

# Call the OCR-enhanced extraction function
documents = extract_text_with_ocr_from_pdf(pdf_file_path)

print(f"Extracted {len(documents)} pages from {pdf_file_path}")
if documents:
    print("First 200 characters of the first page:")
    print(documents[0].page_content[:200])
else:
    print("No text extracted from the PDF, even with OCR. Please check the PDF content.")

Extracted 1 pages from Resume 35.pdf
First 200 characters of the first page:
Is an AI-powered web app that assesses
mental health and provides personalized
therapy suggestions.
AI-based face recognition leverages deep
learning and computer vision to detect and
authenticate fac


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20
)

recursive_docs = recursive_splitter.split_documents(documents)

# PRINT OUTPUT
for i, chunk in enumerate(recursive_docs[:10]):
    print(f"\n🔹 Recursive Chunk {i+1}:")
    print(chunk.page_content)
    print("-" * 50)

print("Total chunks:", len(recursive_docs))


🔹 Recursive Chunk 1:
Is an AI-powered web app that assesses
mental health and provides personalized
therapy suggestions.
AI-based face recognition leverages deep
learning and computer vision to detect and
--------------------------------------------------

🔹 Recursive Chunk 2:
authenticate faces in images or videos.
This AI Cost Calculator estimates AI usage
costs based on inputs like API calls and
tokens.
It helps teams plan budgets and manage AI
expenses effectively.
--------------------------------------------------

🔹 Recursive Chunk 3:
I am a responsible and adaptable person with
good communication skills and a positive attitude.
I am quick to learn, hardworking, and capable of
working both independently and as part of a
--------------------------------------------------

🔹 Recursive Chunk 4:
team. My goal is to gain practical experience and
grow along with the organization.
2023 TO 2025 
SSLC / HSC
Vidhya vikas martic hr.sec
schol 
with 75.86 
Programming Languages and Algorithm

In [ ]:
!pip install faiss-cpu sentence-transformers

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [ ]:
embedding = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

/tmp/ipykernel_506/3751673179.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
vectorstore = FAISS.from_documents(
    recursive_docs,      # 👈 your chunks
    embedding
)

In [ ]:
# For recursive
vectorstore = FAISS.from_documents(recursive_docs, embedding)

In [ ]:
retriever = vectorstore.as_retriever()

In [ ]:
vectorstore.save_local("resume_vector_db")

In [ ]:
query = "What are the skills?"

docs = retriever.invoke(query)

for i, doc in enumerate(docs):
    print(f"\n🔹 Result {i+1}:")
    print(doc.page_content)
    print("-" * 50)


🔹 Result 1:
apply my programming skills and problem-solving abilities. Eager to work on real-world projects, learn modern
technologies, and contribute effectively to a growing organization.
--------------------------------------------------

🔹 Result 2:
frontend development and improve my problem-solving skills.

Machine Learning
--------------------------------------------------

🔹 Result 3:
Dec 2025-Jan 2026

SKILLS

e HTML e Data Structures ¢ SQL ° UI/UX Design

e CSS e Java e Database Management System

e Logical Thinking e Agile Methodologies e Leadership . .
— ; . e Public Speaking
--------------------------------------------------

🔹 Result 4:
. | completed an internship in Machine Learning where | learned basic concepts like data
Euroskillup analysis, model building, and prediction techniques. | gained hands-on experience working
--------------------------------------------------


In [ ]:
print("Total vectors stored:", len(vectorstore.index_to_docstore_id))

Total vectors stored: 17


In [ ]:
def chat_with_resume(query):
    # 🔍 Retrieve relevant chunks
    docs = retriever.invoke(query)

    # 🧠 Combine context
    context = "\n".join([doc.page_content for doc in docs])

    # 🤖 Ask LLM
    response = llm.invoke(
        f"""
        You are a resume assistant.
        Answer only from the given context.

        Context:
        {context}

        Question:
        {query}
        """
    )

    return response.content